
# Vehicle Performance RAG Chatbot  
### Project 2 – Modular Local RAG System (Mistral-7B + CUDA + LangChain + Gradio)

**Purpose:**  
To build a local, GPU-accelerated RAG-based analytics assistant that connects to the SQL database from *Project 1* and supports semantic querying using LangChain, Mistral-7B, and ChromaDB.


In [1]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("CUDA runtime version:", torch.version.cuda)
else:
    print("⚠️ CUDA not detected — check driver installation or restart kernel.")

PyTorch version: 2.5.1+cu121
CUDA available: True
GPU name: NVIDIA GeForce RTX 4060 Ti
CUDA runtime version: 12.1


In [ ]:
# Cell 1: Environment Setup & Imports (Final – Modern LangChain 2026)
# -------------------------------------------------------------------
# Purpose:
# - Import all required modules (updated for modular LangChain v0.3+)
# - Detect GPU availability
# - Initialize project paths
# - Ensure compatibility with modern RAG APIs

import os
import torch
import yaml
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine

# Modern LangChain modules
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import gradio as gr


# Detect device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Paths
PROJECT_ROOT = Path(r"C:\VSCode\vscode_work\resume_projects\DS-AI-VEH-Portfolio\project2_vehicle_performance_rag")
DATA_DIR = PROJECT_ROOT / "data"
CHROMA_DIR = DATA_DIR / "chroma_db"
DOCS_DIR = DATA_DIR / "docs"
CONFIG_PATH = PROJECT_ROOT / "config" / "db_config.yaml"

CHROMA_DIR.mkdir(parents=True, exist_ok=True)


print(f"📁 Project root: {PROJECT_ROOT}")
print(f"📄 Docs path: {DOCS_DIR}")
print(f"💾 Chroma path: {CHROMA_DIR}")
print("✅ Environment initialized successfully.")


In [ ]:

# Cell 2: Load Configuration
# --------------------------
# Purpose:
# - Read database credentials from YAML config file

with open(CONFIG_PATH, "r") as f:
    db_cfg = yaml.safe_load(f)

MYSQL_USER = db_cfg.get("user", "root")
MYSQL_PASSWORD = db_cfg.get("password", "mypassword!")
MYSQL_HOST = db_cfg.get("host", "localhost")
MYSQL_PORT = db_cfg.get("port", 3306)
MYSQL_DB = db_cfg.get("database", "telemetry_analytics_db")

print("Loaded database configuration successfully.")


In [ ]:

# Cell 3: Database Connection Utility
# -----------------------------------
# Purpose:
# - Create reusable SQLAlchemy engine for live analytics queries

def get_db_engine():
    return create_engine(f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}")

engine = get_db_engine()
print(f"Connected to MySQL database: {MYSQL_DB}")


In [ ]:

# Cell 4: Load Reference Documents
# --------------------------------
# Purpose:
# - Load markdown files containing analytics definitions, architecture, data models, etc.

from langchain_core.documents import Document

docs = []
for md_file in DOCS_DIR.glob("*.md"):
    with open(md_file, "r", encoding="utf-8") as f:
        text = f.read()
        docs.append(Document(page_content=text, metadata={"source": md_file.name}))

print(f"Loaded {len(docs)} reference documents.")


In [ ]:

# Cell 5: Vectorization & ChromaDB Store
# --------------------------------------
# Purpose:
# - Split text and create embeddings
# - Store them in a local ChromaDB instance

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
split_docs = splitter.split_documents(docs)

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(split_docs, embedding_model, persist_directory=str(CHROMA_DIR))
vectorstore.persist()

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("ChromaDB vector store created and persisted.")


In [ ]:
# Cell 6: LLM Loader (Mistral-7B + CUDA) — FINAL (HF-safe)
# ------------------------------------------------------
# Purpose:
# - Load local Mistral model
# - HuggingFace pipeline (no stop tokens supported)

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)

generation_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.2,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=generation_pipe)

print("✅ Mistral-7B loaded (HuggingFace-compatible configuration).")


In [ ]:
# Cell 7: RAG Pipeline Construction (FINAL – Stateless, SQL-aware)
# ---------------------------------------------------------------
# Purpose:
# - Stable RAG pipeline for local HuggingFace models
# - Documentation grounding via ChromaDB
# - Prompt explicitly instructs when SQL analytics is required

from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser

# Helper: format retrieved documents into text
def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(doc.page_content for doc in docs)

# Prompt: explicitly tell the LLM about SQL capability
prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are an expert vehicle telemetry analytics assistant.

You have access to:
1. Analytics documentation (definitions, CTQs, architecture)
2. A MySQL database named `telemetry_analytics_db`

Rules:
- If the question is conceptual, answer using documentation context.
- If the question requires numeric aggregation, filtering, time calculation,
  or vehicle-wise comparison, the answer MUST be derived from SQL analytics.
- Never say that data is unavailable if the database contains it.
"""),
    ("human", "Context:\n{context}\n\nQuestion:\n{input}")
])

# Stateless RAG chain (stable for HuggingFacePipeline)
rag_chain = (
    RunnableParallel(
        context=retriever | format_docs,
        input=RunnablePassthrough()
    )
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG pipeline initialized (stateless, SQL-aware).")


In [ ]:
SQL_TEMPLATES = {

    "TOTAL_TIME_IN_REGION": """
        SELECT
            SUM(delta_t) AS total_time
        FROM telemetry_operating_profile
        WHERE source_file = %(vehicle_id)s
          AND rpm_bin_low < %(rpm_high)s
          AND rpm_bin_high > %(rpm_low)s
          AND torque_bin_low < %(torque_high)s
          AND torque_bin_high > %(torque_low)s;
    """,

    "RANK_VEHICLES_IN_REGION": """
        SELECT
            source_file AS vehicle_id,
            SUM(delta_t) AS total_time
        FROM telemetry_operating_profile
        WHERE rpm_bin_low < %(rpm_high)s
          AND rpm_bin_high > %(rpm_low)s
          AND torque_bin_low < %(torque_high)s
          AND torque_bin_high > %(torque_low)s
        GROUP BY source_file
        ORDER BY total_time DESC
        LIMIT %(top_n)s;
    """,

    "RPM_TORQUE_HEATMAP": """
        SELECT
            rpm_bin_label,
            torque_bin_label,
            SUM(delta_t) AS total_time
        FROM telemetry_operating_profile
        WHERE source_file = %(vehicle_id)s
        GROUP BY rpm_bin_label, torque_bin_label
        ORDER BY rpm_bin_label, torque_bin_label;
    """
}


In [ ]:
def classify_intent(question: str) -> str:
    q = question.lower()

    if "heatmap" in q:
        return "RPM_TORQUE_HEATMAP"

    if ("rank" in q or "maximum" in q or "max" in q) and "vehicle" in q:
        return "RANK_VEHICLES_IN_REGION"

    if "total time" in q or "time spent" in q:
        return "TOTAL_TIME_IN_REGION"

    return "RAG_ONLY"


In [ ]:
import re

def extract_parameters(question: str) -> dict:
    params = {}

    vehicle_match = re.search(r"(vehicle[_\s]?\d+)", question, re.IGNORECASE)
    if vehicle_match:
        params["vehicle_id"] = vehicle_match.group(1)

    rpm_match = re.search(r"(\d{4})\s*-\s*(\d{4})", question)
    if rpm_match:
        params["rpm_low"], params["rpm_high"] = map(int, rpm_match.groups())

    torque_match = re.search(r"(\d{2,3})\s*-\s*(\d{2,3})", question)
    if torque_match:
        params["torque_low"], params["torque_high"] = map(int, torque_match.groups())

    params["top_n"] = 1  # default

    return params


In [ ]:
def execute_analytics(intent: str, params: dict) -> str:
    if intent not in SQL_TEMPLATES:
        return "Unsupported analytics intent."

    sql = SQL_TEMPLATES[intent]

    try:
        df = pd.read_sql(sql, engine, params=params)
        if df.empty:
            return "No matching data found."

        try:
            return df.to_markdown(index=False)
        except Exception:
            return df.to_string(index=False)

    except Exception as e:
        return f"SQL execution error:\n{e}"


In [ ]:
def chat_with_rag(message, history):
    try:
        intent = classify_intent(message)

        if intent == "RAG_ONLY":
            return rag_chain.invoke(message)

        params = extract_parameters(message)

        missing = [k for k in ["rpm_low", "rpm_high", "torque_low", "torque_high"] if k not in params]
        if intent != "RPM_TORQUE_HEATMAP" and missing:
            return (
                "Please specify RPM and torque ranges clearly.\n"
                "Example: 2050-2300 RPM and 120-145 Nm."
            )

        if intent in ["TOTAL_TIME_IN_REGION", "RPM_TORQUE_HEATMAP"] and "vehicle_id" not in params:
            return "Please specify a vehicle ID (e.g., Vehicle_02)."

        return execute_analytics(intent, params)

    except Exception as e:
        return f"Error: {str(e)}"


In [14]:

# Cell 9: Gradio Interface
# ------------------------
# Purpose:
# - Build chat interface for the RAG system

def chat_interface(query, history=[]):
    response = chat_with_rag(query)
    history.append((query, response))
    return history, history

chat_ui = gr.ChatInterface(fn=chat_with_rag, title="Vehicle Performance RAG Chatbot", description="Conversational analytics assistant for telemetry-based performance insights.")

chat_ui.launch(debug=True)
